## Tranform Trajectories to Vadere format
This notebook can be used to transform juelich experiment trajectories.
It selectes a measurmenet area, shifts the observation area and transforms the agent positions into the Vadere .traj format. 

Vadere header: pedestrianId simTime endTime startX startY endX endY 
In the end, all observations are in the upper right quadrant of the coordinates (i.e. positive x and y coordinate values only).

Because the data archive was changed after download of the data, each cell also contains the exact revision link for the experiment dokumentation and the access date. 

**Input:** the raw trajectory archives, downloaded from the Jülich data archive (links in `run_overview.csv`, column `perma link`) and unpacked into `data/juelich_db/<experiment>/` (override with the environment variable `JUELICH_DB_DIR`).

**Output:** `data/juelich_trajectories/<scenario>/train/`, the input of step 1 of `batch_run_main.py`.

In [ ]:
import datetime
import os

import numpy as np
import pandas as pd

from dynamics_analysis_with_koopman.config.configs import PROJECT_ROOT

# global settings for data load
# raw archives as downloaded from the Juelich data archive, one folder per experiment
JUELICH_DB_DIR = os.path.join(PROJECT_ROOT, 'data', 'juelich_db')
# converted trajectories, read by step 1 of batch_run_main.py
target_folder = os.path.join(PROJECT_ROOT, 'data', 'juelich_trajectories')

# temporal resolution
FPS_16 = 1 / 16  # 0.0625
FPS_25 = 1 / 25  # 0.04
FPS_30 = 1 / 30  # 0.033..


def load_data_from_file(
    unit,
    root,
    filename,
    temporal_resolution,
    x_shift=0.0,
    y_shift=0.0,
    flip_lr=False,
    flip_ud=False,
    header=['id', 'timeStep', 'x', 'y', 'z'],  # noqa: B006
    target_columns=['pedestrianId', 'simTime', 'endTime', 'startX', 'startY', 'endX', 'endY'],  # noqa: B006
):
    # processing for reading and transforming juelich data files to vadere format
    print('\n\nProcess filename: ', filename)
    dataframe = np.loadtxt(os.path.join(root, filename))
    dataframe = pd.DataFrame(
        dataframe,
        columns=header,
    )

    # rescale to meters
    if unit == 'cm':
        dataframe.x = dataframe.x / 100
        dataframe.y = dataframe.y / 100

    # rotation would be the more suitable transformation
    if flip_lr:
        dataframe.x = (-1) * dataframe.x
        print('Flipped left and right on the x-axis.')

    if flip_ud:
        dataframe.y = (-1) * dataframe.y
        print('Flipped upside down on the y-axis.')

    dataframe.x = dataframe.x + x_shift
    dataframe.y = dataframe.y + y_shift
    print('Applied the x and y shifts.')

    # get minimum x and y values
    x_min = dataframe.x.min()
    y_min = dataframe.y.min()
    print('min_x: ', x_min, 'min_y: ', y_min)
    x_max = dataframe.x.max()
    y_max = dataframe.y.max()
    print('max_x: ', x_max, 'max_y: ', y_max)

    # make timestep integer
    dataframe.timeStep = dataframe.timeStep.astype(int)

    # make x and y float with 3 decimal places
    dataframe.x = dataframe.x.round(3)
    dataframe.y = dataframe.y.round(3)

    # get number of pedestrians as unique id
    num_pedestrians = dataframe.id.nunique()
    print('Number of pedestrians: ', num_pedestrians)

    # transform 'id' to int
    dataframe['pedestrianId'] = dataframe['id'].astype(int)

    # tranform 'timeStep' to float
    dataframe['simTime'] = dataframe['timeStep'] * temporal_resolution

    # get minimum and maximum timeStep
    min_time = dataframe.simTime.min()
    max_time = dataframe.simTime.max()
    print('minimum time in s: ', min_time, 'maximum time in s: ', max_time)

    # offset timeStep
    dataframe.simTime = dataframe.simTime - min_time

    # add the end time and position for each pedestrian as the start position for the next step
    # to do so, we need to sort and then use a simple shift
    dataframe.sort_values(by=['pedestrianId', 'simTime'], inplace=True)
    # for each pedestrian do it one by one (otherwise we will shift data ACROSS pedestrians)
    for ped in dataframe.pedestrianId.unique():
        # get the index of the pedestrian
        idx = dataframe.pedestrianId == ped
        # shift the pedestrian data by one
        dataframe.loc[idx, 'endTime'] = dataframe.loc[idx, 'simTime'].shift(-1)
        dataframe.loc[idx, 'endX'] = dataframe.loc[idx, 'x'].shift(-1)
        dataframe.loc[idx, 'endY'] = dataframe.loc[idx, 'y'].shift(-1)

    # concatenate all the files into one data frame
    dataframe.rename(
        columns={
            'x': 'startX',
            'y': 'startY',
        },
        inplace=True,
    )

    # drop all rows that contain a NaN value (due to shifting)
    dataframe.dropna(inplace=True)

    # sorting for easier review
    # dataframe = dataframe.sort_values(by=['pedestrianId', 'simTime'])  # already sorted by these values
    dataframe = dataframe[target_columns].reset_index(drop=True)

    return dataframe, (num_pedestrians, x_max, y_max, x_min, y_min)


def load_and_process_traj_data(
    root,
    filenames,
    store_dir,
    unit,
    temporal_resolution,
    header=['id', 'timeStep', 'x', 'y', 'z'],  # noqa: B006
    x_shift=0.0,
    y_shift=0.0,
    flip_lr=False,
    flip_ud=False,
):
    print(f'Processed at {datetime.datetime.now()}')

    os.makedirs(store_dir, exist_ok=True)

    global_x_max, global_y_max = -10, -10  # definitely smaller than maximum x, y
    global_x_min, global_y_min = 10, 10  # definitely larger can minimum x, y
    x_max_min, y_max_min = 0, 0
    x_min_max, y_min_max = 99, 99

    for filename in filenames:
        # use offset to make all gris cells positive
        dataframe, (_, x_max, y_max, x_min, y_min) = load_data_from_file(
            unit=unit,
            temporal_resolution=temporal_resolution,
            x_shift=x_shift,
            y_shift=y_shift,
            # to allow cutting off the source in vadere reconstruction
            root=root,
            header=header,
            filename=filename,
            flip_lr=flip_lr,
            flip_ud=flip_ud,
        )

        global_x_max = max(global_x_max, x_max)
        global_y_max = max(global_y_max, y_max)
        global_x_min = min(global_x_min, x_min)
        global_y_min = min(global_y_min, y_min)

        # need this to avoid boundary effects
        x_max_min = max(x_max_min, x_min)
        y_max_min = max(y_max_min, y_min)
        x_min_max = min(x_min_max, x_max)
        y_min_max = min(y_min_max, y_max)

        with open(os.path.join(store_dir, f'{filename}'), 'wb') as f:
            dataframe.to_csv(f, sep=' ', header=True, index=False)

    print('\n\n\n=====')
    print(f'All data within the following measurement area:')
    print(f'x in [{global_x_min}, {global_x_max}], y in [{global_y_min}, {global_y_max}]')
    print('=====')
    print(f'Intersection with data available in all runs:')
    print(f'x in [{x_max_min}, {x_min_max}], y in [{y_max_min}, {y_min_max}]')
    print('=====')


def rename_files_by_group_schema(store_dir, filenames, group_suffix_map):
    # renames stored output files to match the archive's updated naming schema
    # (e.g. '_a_' -> '_a1_'), applied once per group, in-place in store_dir
    for filename in filenames:
        new_filename = filename
        for old, new in group_suffix_map.items():
            if old in new_filename:
                new_filename = new_filename.replace(old, new)
                break
        if new_filename != filename:
            os.rename(os.path.join(store_dir, filename), os.path.join(store_dir, new_filename))

In [ ]:
root = os.path.join(JUELICH_DB_DIR, '2006bottleneck/2006bottleneck2_trajectories')
filenames = [
    'b090_combined.txt',
    'b100_combined.txt',
    'b110_combined.txt',
    'b120_combined.txt',
    'b140_combined.txt',
    'b160_combined.txt',
    'b180_combined.txt',
    'b200_combined.txt',
    'b220_combined.txt',
    'b250_combined.txt',
]

unit = 'cm'
temporal_resolution = FPS_25
store_dir = os.path.join(target_folder, '2006bottleneck', 'train')

load_and_process_traj_data(
    root=root,
    filenames=filenames,
    store_dir=store_dir,
    unit=unit,
    temporal_resolution=temporal_resolution,
    x_shift=2.5,
    y_shift=3,
    header=['id', 'timeStep', 'x', 'y'],
)

In [ ]:
root = os.path.join(JUELICH_DB_DIR, '2009corner/2009corner_open_trajectories_txt')
filenames = [  # width of 240cm
    'eo-240-050-240.txt',
    'eo-240-060-240.txt',
    'eo-240-080-240.txt',
    'eo-240-100-240.txt',
    'eo-240-150-240.txt',
    'eo-240-240-240.txt',
    # width of 300cm
    'eo-300-050-300.txt',
    'eo-300-060-300.txt',
    'eo-300-080-300.txt',
    'eo-300-100-300.txt',
    'eo-300-150-300.txt',
    #  'eo-300-300-300-bi.txt', # did not make it to selection parameters
    'eo-300-300-300.txt',
]

unit = 'cm'
temporal_resolution = FPS_16
store_dir = os.path.join(target_folder, '2009corner', 'train')

load_and_process_traj_data(
    root=root,
    filenames=filenames,
    store_dir=store_dir,
    unit=unit,
    x_shift=3,
    y_shift=3,
    temporal_resolution=temporal_resolution,
)

In [ ]:
root = os.path.join(JUELICH_DB_DIR, '2009tjunction/2009tjunction_trajectories_txt')
filenames = [
    'ko-240-050-240.txt',
    'ko-240-060-240.txt',
    'ko-240-080-240.txt',
    'ko-240-100-240.txt',
    'ko-240-120-240.txt',
    'ko-240-150-240.txt',
    'ko-240-240-240.txt',
    'ko-300-050-300.txt',
    'ko-300-080-300.txt',
    'ko-300-120-300.txt',
    'ko-300-150-300.txt',
]

unit = 'cm'
temporal_resolution = FPS_16
store_dir = os.path.join(target_folder, '2009tjunction', 'train')

load_and_process_traj_data(
    root=root,
    filenames=filenames,
    store_dir=store_dir,
    unit=unit,
    temporal_resolution=temporal_resolution,
    x_shift=5,
    y_shift=3,
)

In [ ]:
root = os.path.join(JUELICH_DB_DIR, '2009unidirClosed/2009unidirectional_closed_trajectories_txt')
filenames = [
    'ug-100-007.txt',
    'ug-100-015.txt',
    'ug-100-030.txt',
    'ug-100-045.txt',
    'ug-100-060.txt',
    'ug-100-075.txt',
    'ug-100-090.txt',
    'ug-100-120.txt',
    'ug-140-010.txt',
    'ug-140-020.txt',
    'ug-140-045.txt',
    'ug-140-065.txt',
    'ug-140-085.txt',
    'ug-140-110.txt',
    'ug-140-130.txt',
    'ug-180-015.txt',
    'ug-180-030.txt',
    'ug-180-060.txt',
    'ug-180-085.txt',
    'ug-180-095.txt',
    'ug-180-110.txt',
    'ug-180-110v2.txt',
    'ug-180-140.txt',
    'ug-180-230.txt',
]

unit = 'cm'
temporal_resolution = FPS_16
store_dir = os.path.join(target_folder, '2009unidirClosed', 'train')

load_and_process_traj_data(
    root=root,
    filenames=filenames,
    store_dir=store_dir,
    unit=unit,
    temporal_resolution=temporal_resolution,
    x_shift=1,  # corridor width from [1, b], b = {1, 1.4, 1.8}
    y_shift=3,  # this moves the corridor length to [0, 6]
)

In [ ]:
root = os.path.join(JUELICH_DB_DIR, '2009unidirOpen/2009unidirectional_open_trajectories_txt')
filenames = [  # Bottleneck
    'uo-300-300-200.txt',
    'uo-300-300-160.txt',
    'uo-300-300-120.txt',
    'uo-300-300-080.txt',
    'uo-240-240-160.txt',
    'uo-240-240-130.txt',
    'uo-240-240-100.txt',
    'uo-180-180-120.txt',
    'uo-180-180-095.txt',
    'uo-180-180-070.txt',
]

unit = 'cm'
temporal_resolution = FPS_16
store_dir = os.path.join(target_folder, '2009unidirOpenB', 'train')

load_and_process_traj_data(
    root=root,
    filenames=filenames,
    store_dir=store_dir,
    unit=unit,
    temporal_resolution=temporal_resolution,
    x_shift=1,
    y_shift=7,  # this moves bottleneck to 3 (former -4)
)

In [ ]:
root = os.path.join(JUELICH_DB_DIR, '2009unidirOpen/2009unidirectional_open_trajectories_txt')
filenames = [
    # Corridor with varying widths
    'uo-300-300-300.txt',
    'uo-240-300-300.txt',
    'uo-240-240-240.txt',
    'uo-190-240-240.txt',
    'uo-180-300-300.txt',
    'uo-180-180-180.txt',
    'uo-145-240-240.txt',
    'uo-145-180-180.txt',
    'uo-120-300-300.txt',
    'uo-100-300-300.txt',
    'uo-100-180-180.txt',
    'uo-095-240-240.txt',
    'uo-080-300-300v2.txt',
    'uo-080-240-240.txt',
    'uo-070-180-180.txt',
    'uo-065-240-240.txt',
    'uo-060-180-180.txt',
    'uo-050-180-180.txt',
]

unit = 'cm'
temporal_resolution = FPS_16
store_dir = os.path.join(target_folder, '2009unidirOpenC', 'train')

load_and_process_traj_data(
    root=root,
    filenames=filenames,
    store_dir=store_dir,
    unit=unit,
    temporal_resolution=temporal_resolution,
    x_shift=2,
    y_shift=7,  # this moves bottleneck to 3 (former -4)
)

In [ ]:
root = os.path.join(JUELICH_DB_DIR, '2013bidirectional/BI_CORR/BI_CORR_400_A')
filenames = [
    'BI_CORR_400_A_1.txt',
    'BI_CORR_400_A_2.txt',
    'BI_CORR_400_A_3.txt',
    'BI_CORR_400_A_4.txt',
    'BI_CORR_400_A_5.txt',
]

unit = 'm'
# File names with small letters were extracted with 25 frames per second
# and file names with capital letters were extracted with 16 frames per second.
temporal_resolution = FPS_16
store_dir = os.path.join(target_folder, '2013bidirectional_A', 'train')

load_and_process_traj_data(
    root=root,
    filenames=filenames,
    store_dir=store_dir,
    unit=unit,
    temporal_resolution=temporal_resolution,
    x_shift=3,
    y_shift=8,
    header=['id', 'timeStep', 'x', 'y', 'z', 'rot', 'markerHead', 'flag'],
)

In [ ]:
root = os.path.join(JUELICH_DB_DIR, '2013bidirectional/BI_CORR/BI_CORR_400_B')
filenames = [
    'bi_corr_400_b_01.txt',
    'bi_corr_400_b_02.txt',
    'bi_corr_400_b_03.txt',
    'bi_corr_400_b_04.txt',
    'bi_corr_400_b_05.txt',
    'bi_corr_400_b_06.txt',
    'bi_corr_400_b_07.txt',
    'bi_corr_400_b_08.txt',
    'bi_corr_400_b_09.txt',
    'bi_corr_400_b_10.txt',
]

unit = 'cm'
# File names with small letters were extracted with 25 frames per second
# and file names with capital letters were extracted with 16 frames per second.
temporal_resolution = FPS_25
store_dir = os.path.join(target_folder, '2013bidirectional_B', 'train')

load_and_process_traj_data(
    root=root,
    filenames=filenames,
    store_dir=store_dir,
    unit=unit,
    temporal_resolution=temporal_resolution,
    x_shift=6.5,
    y_shift=1,
)

In [ ]:
root = os.path.join(JUELICH_DB_DIR, '2013crossing90/2013crossing_90_trajectories_txt')
filenames = [
    'crossing_90_a_01.txt',
    'crossing_90_a_02.txt',
    'crossing_90_a_03.txt',
    'crossing_90_a_04.txt',
    'crossing_90_a_05.txt',
    'crossing_90_a_06.txt',
    'crossing_90_a_07.txt',
    'crossing_90_a_08.txt',
    'crossing_90_a_09.txt',
    'crossing_90_a_10.txt',
    'crossing_90_b_02.txt',
    'crossing_90_b_03.txt',
    'crossing_90_b_04.txt',
    'crossing_90_b_05.txt',
    'crossing_90_b_06.txt',
    'crossing_90_b_07.txt',
    'crossing_90_c_01.txt',
    'crossing_90_c_02.txt',
    'crossing_90_c_03.txt',
    'crossing_90_c_04.txt',
    'crossing_90_c_05.txt',
    'crossing_90_c_06.txt',
    'crossing_90_c_07.txt',
    'crossing_90_c_08.txt',
]

unit = 'cm'
temporal_resolution = FPS_25
store_dir = os.path.join(target_folder, '2013crossing90', 'train')

load_and_process_traj_data(
    root=root,
    filenames=filenames,
    store_dir=store_dir,
    unit=unit,
    temporal_resolution=temporal_resolution,
    x_shift=3,
    y_shift=0.5,
)


rename_files_by_group_schema(store_dir, filenames, {'_a_': '_a1_', '_b_': '_b2_', '_c_': '_c3_'})

In [ ]:
root = os.path.join(JUELICH_DB_DIR, '2013crossing90/2013crossing_90_trajectories_txt')
# leave out the files with "clean" observation area being relatively small
crossing_upper_right = [
    'CROSSING_90_D_1.txt',
    #  'CROSSING_90_D_2.txt',
    'CROSSING_90_D_3.txt',
    'CROSSING_90_D_5.txt',
    #  'CROSSING_90_D_6.txt',
    'CROSSING_90_D_7.txt',
    #  'CROSSING_90_D_8.txt',
    'CROSSING_90_E_1.txt',
    #  'CROSSING_90_E_6.txt',
    'CROSSING_90_F_1.txt',
    'CROSSING_90_F_3.txt',
    'CROSSING_90_G_1.txt',
    'CROSSING_90_G_2.txt',
    'CROSSING_90_G_3.txt',
]

unit = 'm'
temporal_resolution = FPS_16
store_dir = os.path.join(target_folder, '2013crossing90', 'train')

load_and_process_traj_data(
    root=root,
    filenames=crossing_upper_right,
    store_dir=store_dir,
    unit=unit,
    temporal_resolution=temporal_resolution,
    x_shift=8,
    y_shift=7,
    header=['id', 'timeStep', 'x', 'y', 'z', 'rot', 'markerHead', 'flag'],
)

# flip upside down and left-right for 180 rotation
crossing_lower_left = [
    'CROSSING_90_D_2.txt',
    'CROSSING_90_D_6.txt',
    'CROSSING_90_D_8.txt',
    'CROSSING_90_E_6.txt',
]

load_and_process_traj_data(
    root=root,
    filenames=crossing_lower_left,
    store_dir=store_dir,
    unit=unit,
    temporal_resolution=temporal_resolution,
    x_shift=8,
    y_shift=9,
    flip_lr=True,
    flip_ud=True,
    header=['id', 'timeStep', 'x', 'y', 'z', 'rot', 'markerHead', 'flag'],
)

rename_files_by_group_schema(
    store_dir, crossing_upper_right + crossing_lower_left, {'_D_': '_D4_', '_E_': '_E5_', '_F_': '_F6_', '_G_': '_G7_'}
)

In [ ]:
root = os.path.join(JUELICH_DB_DIR, '2013crossing120/2013crossing_120_trajectories_txt')
filenames = [
    'crossing_120_a_01.txt',
    'crossing_120_a_02.txt',
    'crossing_120_a_03.txt',
    'crossing_120_a_04.txt',
    'crossing_120_a_05.txt',
    'crossing_120_a_06.txt',
    'crossing_120_a_07.txt',
    'crossing_120_b_01.txt',
    'crossing_120_b_02.txt',
    'crossing_120_c_01.txt',
    'crossing_120_c_02.txt',
    'crossing_120_c_03.txt',
    'crossing_120_c_04.txt',
    'crossing_120_c_05.txt',
    'crossing_120_c_06.txt',
    'crossing_120_c_07.txt',
]

unit = 'm'
temporal_resolution = FPS_16
store_dir = os.path.join(target_folder, '2013crossing120', 'train')

load_and_process_traj_data(
    root=root,
    filenames=filenames,
    store_dir=store_dir,
    unit=unit,
    temporal_resolution=temporal_resolution,
    x_shift=5,
    y_shift=3,
    header=['id', 'timeStep', 'x', 'y', 'z', 'rot', 'markerHead', 'flag'],
)


rename_files_by_group_schema(store_dir, filenames, {'_a_': '_a1_', '_b_': '_b2_', '_c_': '_c3_'})

In [ ]:
root = os.path.join(JUELICH_DB_DIR, '2013singleFile/PHAS_SEP')
filenames = [
    'PHASE_SEP_1_pos.csv',
    'PHASE_SEP_2_pos.csv',
    'PHASE_SEP_3_pos.csv',
    'PHASE_SEP_4_pos.csv',
    'PHASE_SEP_5_pos.csv',
    'PHASE_SEP_6_pos.csv',
]

unit = 'm'
temporal_resolution = FPS_16
store_dir = os.path.join(target_folder, '2013singleFile', 'train')

load_and_process_traj_data(
    root=root,
    filenames=filenames,
    store_dir=store_dir,
    unit=unit,
    temporal_resolution=temporal_resolution,
    x_shift=5,
    y_shift=6,
)

In [ ]:
root = os.path.join(JUELICH_DB_DIR, '2013unidirectional')

unit = 'm'
temporal_resolution = FPS_25
store_dir = os.path.join(target_folder, '2013unidirectionalC', 'train')

# starting left already walks the direction we want
corridor_start_left = [  #'traj_UNI_CORR_500_01.txt',
    #'traj_UNI_CORR_500_02.txt',
    'traj_UNI_CORR_500_03.txt',
    # 'traj_UNI_CORR_500_04.txt',
    'traj_UNI_CORR_500_05.txt',
]
load_and_process_traj_data(
    root=root,
    filenames=corridor_start_left,
    store_dir=store_dir,
    unit=unit,
    temporal_resolution=temporal_resolution,
    x_shift=6,
    y_shift=1,
)

# starting rights needs to be flipped
corridor_start_right = [
    'traj_UNI_CORR_500_01.txt',
    'traj_UNI_CORR_500_02.txt',
    # 'traj_UNI_CORR_500_03.txt',
    'traj_UNI_CORR_500_04.txt',
    # 'traj_UNI_CORR_500_05.txt',
]
load_and_process_traj_data(
    root=root,
    filenames=corridor_start_right,
    store_dir=store_dir,
    unit=unit,
    temporal_resolution=temporal_resolution,
    x_shift=5,
    y_shift=1,
    flip_lr=True,
)

In [ ]:
root = os.path.join(JUELICH_DB_DIR, '2013unidirectional')

unit = 'm'
temporal_resolution = FPS_25
store_dir = os.path.join(target_folder, '2013unidirectionalB', 'train')

# already the direction we want
bottleneck_start_left = [  #'traj_UNI_CORR_500_06.txt',
    'traj_UNI_CORR_500_07.txt',
    #'traj_UNI_CORR_500_08.txt',
    'traj_UNI_CORR_500_09.txt',
]

load_and_process_traj_data(
    root=root,
    filenames=bottleneck_start_left,
    store_dir=store_dir,
    unit=unit,
    temporal_resolution=temporal_resolution,
    x_shift=6,
    y_shift=1,
)

# needs to be flipped along x axis
bottleneck_start_right = [
    'traj_UNI_CORR_500_06.txt',
    #'traj_UNI_CORR_500_07.txt',
    'traj_UNI_CORR_500_08.txt',
    #'traj_UNI_CORR_500_09.txt'
]

load_and_process_traj_data(
    root=root,
    filenames=bottleneck_start_right,
    store_dir=store_dir,
    unit=unit,
    temporal_resolution=temporal_resolution,
    x_shift=6,
    y_shift=1,
    flip_lr=True,
)

In [ ]:
root = os.path.join(JUELICH_DB_DIR, '2017sime/2017sime_trajectories_txt')
filenames = [
    '1501.txt',
    '1502.txt',
    '1503.txt',
    '1504.txt',
    '1507.txt',
]

unit = 'm'
temporal_resolution = FPS_25
store_dir = os.path.join(target_folder, '2017simeB', 'train')

load_and_process_traj_data(
    root=root,
    filenames=filenames,
    store_dir=store_dir,
    unit=unit,
    temporal_resolution=temporal_resolution,
    x_shift=16,
    y_shift=3,
)

In [ ]:
root = os.path.join(JUELICH_DB_DIR, '2017sime/2017sime_trajectories_txt')
filenames = ['1601.txt', '1602.txt', '1603.txt', '1604.txt', '1605.txt', '1607.txt', '1608.txt']

unit = 'm'
temporal_resolution = FPS_25
store_dir = os.path.join(target_folder, '2017simeC', 'train')

load_and_process_traj_data(
    root=root,
    filenames=filenames,
    store_dir=store_dir,
    unit=unit,
    temporal_resolution=temporal_resolution,
    x_shift=16,
    y_shift=3,
)

In [ ]:
root = os.path.join(
    JUELICH_DB_DIR, '2022wheelchair_luggage_bottleneck/2022wheelchair_luggage_bottleneck_trajectories_txt'
)
filenames = [
    # baseline (24 participants)
    '1AM030.txt',
    '1AM050.txt',
    '1AM090.txt',
    '1AM110.txt',
    '1AM130.txt',
    '1AM150.txt',
    '1AM160.txt',
    '1AM180.txt',
    '1AM200.txt',
    '1AM230.txt',
    '1AM250.txt',
    '1AM270.txt',
    '1AM300.txt',
    '1AM320.txt',
    '1AM340.txt',
    # urgent (24 participants)
    '1AM020.txt',
    '1AM040.txt',
    '1AM060.txt',
    '1AM080.txt',
    '1AM100.txt',
    '1AM120.txt',
    '1AM140.txt',
    '1AM170.txt',
    '1AM190.txt',
    '1AM210.txt',
    '1AM220.txt',
    '1AM240.txt',
    '1AM260.txt',
    '1AM280.txt',
    '1AM290.txt',
    '1AM310.txt',
    '1AM330.txt',
    '1AM350.txt',
]

unit = 'm'
temporal_resolution = FPS_30
store_dir = os.path.join(target_folder, '2022wheelchair_luggage_bottleneck', 'train')

load_and_process_traj_data(
    root=root,
    filenames=filenames,
    store_dir=store_dir,
    unit=unit,
    temporal_resolution=temporal_resolution,
    x_shift=7,
    y_shift=3,
    header=['id', 'timeStep', 'x', 'y', 'z', 'markerHead'],
)